# help! — pi0.5 LoRA fine-tune on Colab

Fine-tunes pi0.5 (`pi05_base`, LoRA) on one help! task's expert demos with upstream openpi and
`helpsim/openpi_help.py` (TASK_SPEC.md §2–3, §7). Runtime: LoRA needs more than 22.5 GB. **A100** (batch 32), or the **v6e-1 TPU** (32 GB, batch 32; the install
switches openpi to `jax[tpu]`), or an **L4** with `BATCH = 16`. Not the v5e-1 TPU (16 GB) or the G4 (openpi's
pinned JAX does not support that GPU).

**Order:** shelf first, then drawer. For each task, set `TASK` and its unit budget in **Settings**, then run every cell
from the top.

1. **Smoke run** (about 5 units): 120 updates. It checks that the loss falls and a checkpoint saves, measures seconds
   per update, and sizes the full run (`STEPS`) to your budget.
2. **Full run.** Checkpoints are written to local disk; one is copied to Drive halfway and one at the end
   (`DRIVE_EVERY`). After a disconnect, run the cells again from the top: the run resumes from the copy on Drive.
3. **Open-loop check** on held-out demos (the `_val` dataset, never trained on).
4. **Serve** the policy for evaluation in the sim on the Mac.

Keep this tab open during training: without Colab Pro+ background execution, closing it can end the runtime.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo "no GPU (TPU runtime?): $(ls -d /dev/accel* /dev/vfio/[0-9]* 2>/dev/null | tr '\n' ' ')"


## Settings

`UNITS_PER_HOUR`: read it from Colab's resources panel (Runtime → View resources) for your runtime.
`BUDGET_UNITS` is what the full run may spend; the smoke run turns it into a number of updates.
Suggested split of 200 units: shelf 80, drawer 60, and the rest for smoke runs and serving.

The project folder is the Stanford Drive's `stanford/classes/cs238/help-your-robot`. If Colab runs under another
account, share that folder with it (Editor) and add a shortcut to it in that account's My Drive. Files the notebook
writes there count against the Colab account's storage, and files it deletes stay in that account's trash until it
is emptied: a checkpoint is about 7 GB.

**Without Drive** (`USE_DRIVE = False`): mounting Drive needs a sign-in popup in the browser. Without it, the
datasets come from public copies (`data/tars/` on Drive, readable by anyone with the link), everything is written to
local disk, and the cell after **Full run** serves `ROOT` over a Cloudflare quick tunnel so that
`scripts/pull_ckpt.sh` on the Mac copies norm stats, logs and each checkpoint the run selects to Drive (`pi05/`).
Anyone with that URL can download them while it runs. To resume on a fresh runtime, set `RESTORE` to the file id
`pull_ckpt.sh` prints: the notebook fetches that checkpoint from Drive's public link.

In [ ]:
import os, json, subprocess

USE_DRIVE = True          # False: no Drive mount (it needs a sign-in popup); see "Without Drive" above
RESTORE = ''              # without Drive: the restore id from scripts/pull_ckpt.sh, to resume on a fresh runtime
TASK = 'shelf'            # 'shelf' or 'drawer'
BUDGET_UNITS = 80         # for the full run of this task
UNITS_PER_HOUR = 13.0     # A100 13, v6e-1 TPU about 5, L4 about 2; check Runtime -> View resources
BATCH = 32                # 16 on an L4, or if a 40 GB A100 runs out of memory
MAX_STEPS = 30000         # the full run never goes past this (20000 on an L4 keeps it under a day)
EXP = 'run1'              # a new name starts a new run; the same name resumes
KEEP_PERIOD = 5000        # local disk: also keep every n-th checkpoint; 0 keeps only the latest
DRIVE_EVERY = 0           # copy a checkpoint (about 7 GB) to Drive every n updates and at the end; 0: halfway and end

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # The project's folder: on the Stanford Drive, or a shortcut to it in another account's My Drive.
    PROJECT = next((p for p in ('/content/drive/MyDrive/stanford/classes/cs238/help-your-robot',
                                '/content/drive/MyDrive/help-your-robot') if os.path.isdir(p)), None)
    assert PROJECT, 'help-your-robot not found in My Drive: mount the Stanford Drive or add a shortcut (see above)'
else:
    PROJECT = '/content/project'             # local disk; scripts/pull_ckpt.sh copies pi05/ to Drive
DATA = f'{PROJECT}/data'                     # collected runs and lerobot/ datasets (scripts/push_to_drive.sh)
ROOT = f'{PROJECT}/pi05'                     # assets/ (norm stats), checkpoints/, logs/, jax_cache/
REPO = f'help/{TASK}_pi05'
CKPT_LOCAL = '/content/checkpoints'
CFG = f'help_{TASK}_pi05_lora'               # the openpi config name (helpsim/openpi_help.py)
OPENPI_COMMIT = '215abfb217dbac7d5f1273282331b9b1866c0479'   # helpsim/openpi_help.py OPENPI_COMMIT

# Public copies of the datasets on Drive (data/tars/, anyone with the link can read): Drive file ids.
DATA_TARS = {'shelf_pi05': '19KEfEmtIaA8hnT8TqAvNIRgr9MUCnm8W', 'shelf_pi05_val': '1s9e2oz5USwZ02CfstdYyifjyH4ijeS6Y',
             'drawer_pi05': '1xvJFwx0sf4l_XTLvNmyhdPNj8XgVlQhT', 'drawer_pi05_val': '1jnXBe5nPcR7BP-vDVgx5k6EWJmv1o_D_'}
PUBLIC = 'https://drive.usercontent.google.com/download?export=download&confirm=t&id='
if not USE_DRIVE:
    os.makedirs(f'{DATA}/lerobot', exist_ok=True)
    for r in (REPO, f'{REPO}_val'):
        if not os.path.isdir(f'{DATA}/lerobot/{r}'):   # each tar holds help/<name>/
            subprocess.run(f"curl -sfL '{PUBLIC}{DATA_TARS[r.split('/')[1]]}' | tar -x -C {DATA}/lerobot",
                           shell=True, check=True)
assert os.path.isdir(f'{DATA}/lerobot/{REPO}'), f'dataset not on Drive yet: {DATA}/lerobot/{REPO}'
for d in ('assets', 'checkpoints', 'logs'):
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)
os.makedirs(f'{CKPT_LOCAL}/{CFG}/{EXP}', exist_ok=True)
if not USE_DRIVE and RESTORE and not os.listdir(f'{CKPT_LOCAL}/{CFG}/{EXP}'):
    # pull_ckpt.sh's manifest: {'step': n, 'files': {path under pi05/: Drive file id}}
    m = json.loads(subprocess.run(f"curl -sfL '{PUBLIC}{RESTORE}'", shell=True, check=True,
                                  capture_output=True, text=True).stdout)
    jobs = ''.join(f"{fid} {ROOT}/{rel}\n" for rel, fid in m['files'].items())
    subprocess.run(f"xargs -P 8 -L 1 sh -c 'mkdir -p \"$(dirname $1)\" && curl -sfL -o $1 \"{PUBLIC}$0\"'",
                   shell=True, check=True, input=jobs, text=True)
    print('restored step', m['step'], f"({len(m['files'])} files) from Drive")
# openpi's train.py caches compiled XLA programs in ~/.cache/jax, on local disk: a new runtime compiles again.
os.environ.update(USE_DRIVE=str(int(USE_DRIVE)), TASK=TASK, DATA=DATA, ROOT=ROOT, REPO=REPO, EXP=EXP, CFG=CFG, BATCH=str(BATCH),
                  KEEP_PERIOD=str(KEEP_PERIOD), CKPT_LOCAL=CKPT_LOCAL, OPENPI_COMMIT=OPENPI_COMMIT,
                  HF_LEROBOT_HOME='/content/lerobot', XLA_PYTHON_CLIENT_MEM_FRACTION='0.9', PYTHONUNBUFFERED='1',
                  UV_NO_SYNC='1')   # uv run keeps the env as installed (on a TPU, jax[tpu] replaces the CUDA plugins)
RUN = 'cd /content/openpi && uv run python /content/help-your-robot/scripts/openpi_run.py'
TIMER = 'python3 /content/help-your-robot/scripts/step_timer.py'   # standard library only
os.environ.update(RUN=RUN, TIMER=TIMER)
print(TASK, REPO, '->', ROOT)
!df -h {PROJECT} | tail -1

## Install

Upstream openpi at the pinned commit, in its own `uv` environment, plus this repo. About 5 minutes on a fresh runtime;
nothing is reinstalled if the runtime is still up. It prints the devices JAX sees: check it is the GPU or TPU.

In [ ]:
%%bash
set -e
cd /content
[ -d openpi ] || git clone -q https://github.com/Physical-Intelligence/openpi.git
cd openpi
git fetch -q origin "$OPENPI_COMMIT" 2>/dev/null || true
git checkout -q "$OPENPI_COMMIT" && git submodule update -q --init --recursive
command -v uv >/dev/null || pip -q install uv
GIT_LFS_SKIP_SMUDGE=1 uv sync -q
# A TPU runtime (no GPU): swap the CUDA build of jax for the TPU one, same version.
if ! command -v nvidia-smi >/dev/null && ls /dev/accel* /dev/vfio/[0-9]* >/dev/null 2>&1; then
  uv pip uninstall -q --python .venv/bin/python jax-cuda12-plugin jax-cuda12-pjrt 2>/dev/null || true
  uv pip install -q --python .venv/bin/python "jax[tpu]==$(.venv/bin/python -c 'import jax; print(jax.__version__)')" \
      -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
fi
.venv/bin/python -c "import jax; print('jax', jax.__version__, jax.devices())"
cd /content
if [ -d help-your-robot ]; then git -C help-your-robot pull -q; else git clone -q https://github.com/DarinAnthony/help-your-robot.git; fi
echo "openpi $(git -C openpi rev-parse --short HEAD), help-your-robot $(git -C help-your-robot rev-parse --short HEAD)"

## Data

Copy the training and held-out datasets to local disk: decoding video from the Drive mount is slow.

In [ ]:
%%bash
set -e
mkdir -p /content/lerobot/help
for r in "$REPO" "${REPO}_val"; do
  if [ -d "$DATA/lerobot/$r" ]; then
    rsync -a "$DATA/lerobot/$r" /content/lerobot/help/
    python -c "import json; d=json.load(open('/content/lerobot/$r/meta/info.json')); print('$r:', d['total_episodes'], 'episodes,', d['total_frames'], 'frames')"
  else
    echo "$r: not on Drive"
  fi
done
du -sh /content/lerobot/help

## Normalization statistics

Computed once per dataset and kept on Drive (`ROOT/assets`). Set `REDO = True` after the dataset changes.

In [ ]:
REDO = False
stats = f'{ROOT}/assets/{CFG}/{REPO}/norm_stats.json'
if REDO or not os.path.isfile(stats):
    !$RUN norm-stats $TASK --repo $REPO --root $ROOT 2>&1 | grep -v -i warn | tail -3
s = json.load(open(stats))['norm_stats']
print('state q01', [round(x, 2) for x in s['state']['q01']])
print('actions q01', [round(x, 2) for x in s['actions']['q01'][:11]])

## Smoke run

120 updates with checkpoints on local disk (not Drive). It downloads `pi05_base` (a few minutes) and compiles
(a few minutes, then cached on Drive). Check that the loss falls. The timing excludes the first logged interval
(compilation) and sets `STEPS` for the full run.

In [ ]:
%%bash
set -o pipefail
$RUN train $TASK --repo $REPO --root $ROOT --ckpt-root /content/smoke --exp smoke --steps 120 --batch-size $BATCH \
    --log-interval 10 --save-interval 100 --keep-period 0 --overwrite 2>&1 \
  | $TIMER --steps 10000 --quiet --json /content/smoke_timing.json
ls /content/smoke/$CFG/smoke && du -sh /content/smoke/$CFG/smoke/*

In [ ]:
t = json.load(open('/content/smoke_timing.json'))
sec = t['sec_per_update']
STEPS = int(BUDGET_UNITS / UNITS_PER_HOUR * 3600 / sec) // 1000 * 1000
STEPS = max(2000, min(STEPS, MAX_STEPS))
hours = STEPS * sec / 3600
print(f'{sec:.2f} s/update at batch {BATCH} -> STEPS = {STEPS}: {hours:.1f} h, about {hours * UNITS_PER_HOUR:.0f} units')
print(f'that is {STEPS * BATCH:,} samples')
os.environ['STEPS'] = str(STEPS)
os.environ['DRIVE_EVERY'] = str(DRIVE_EVERY or max(1000, STEPS // 2000 * 1000))
print('checkpoints copied to Drive at multiples of', os.environ['DRIVE_EVERY'], 'and at', STEPS)

## Full run

Resumes from the checkpoint on Drive if there is one (same `EXP`). Checkpoints are written locally every 1,000
updates. A background loop checks every 5 minutes and copies to `ROOT/checkpoints` only the newest finished
checkpoint at a multiple of `DRIVE_EVERY` (or the last step), then removes the older copy from Drive. The log goes
to `ROOT/logs`.

To change `STEPS` for a run that has started, keep the same value: the learning-rate schedule depends on it.

In [ ]:
%%bash
set -e
L="$CKPT_LOCAL/$CFG/$EXP"; D="$ROOT/checkpoints/$CFG/$EXP"
mkdir -p "$L" "$D"
# Resume: bring the Drive copy back to a fresh runtime.
if [ -z "$(ls -A "$L")" ] && [ -n "$(ls "$D")" ]; then
  echo "restoring $(ls "$D" | tr '\n' ' ')from Drive"; rsync -a "$D/" "$L/"
fi
cat > /content/sync_ckpt.sh <<'EOF'
#!/bin/bash
# Copy the newest finished checkpoint at a multiple of DRIVE_EVERY (or the last step) to Drive, then remove older
# copies there. Orbax writes a checkpoint under a temporary name and renames it to the bare step number when complete.
L="$CKPT_LOCAL/$CFG/$EXP"; D="$ROOT/checkpoints/$CFG/$EXP"
sync_once() {
  last=$(ls "$L" | grep -E '^[0-9]+$' | awk -v e="$DRIVE_EVERY" -v s="$STEPS" '$1 % e == 0 || $1 == s' | sort -n | tail -1)
  [ -n "$last" ] && [ -d "$L/$last/params" ] || return 0
  [ -e "$D/.$last.done" ] && return 0
  rsync -a --whole-file "$L/$last/" "$D/$last/" && touch "$D/.$last.done" || return 0
  for old in $(ls "$D" | grep -E '^[0-9]+$'); do
    [ "$old" = "$last" ] || { rm -rf "$D/$old"; rm -f "$D/.$old.done"; }
  done
  echo "$(date) on Drive: $last"
}
if [ "$1" = once ]; then sync_once; exit; fi
while true; do sync_once; sleep 300; done
EOF
chmod +x /content/sync_ckpt.sh
pkill -f sync_ckpt.sh || true
nohup /content/sync_ckpt.sh > /content/sync_ckpt.log 2>&1 &
echo "copying checkpoints at multiples of $DRIVE_EVERY and step $STEPS: $L -> $D"


In [ ]:
%%bash
# Without Drive: serve ROOT (norm stats, logs, the checkpoint copies) over a quick tunnel for scripts/pull_ckpt.sh.
[ "$USE_DRIVE" = 1 ] && { echo 'Drive is mounted: nothing to serve'; exit 0; }
CF=/content/cloudflared
[ -x $CF ] || { wget -q -O $CF https://github.com/cloudflare/cloudflared/releases/download/2025.8.1/cloudflared-linux-amd64 && chmod +x $CF; }
pgrep -f 'http.server 8001' >/dev/null || \
  nohup python3 -m http.server 8001 --bind 127.0.0.1 --directory "$ROOT" > /content/files.log 2>&1 &
pgrep -f 'localhost:8001' >/dev/null || \
  nohup $CF tunnel --no-autoupdate --url http://localhost:8001 > /content/files_tunnel.log 2>&1 &
for i in $(seq 1 30); do grep -qo 'https://[a-z0-9-]*\.trycloudflare\.com' /content/files_tunnel.log && break; sleep 2; done
echo "on the Mac: scripts/pull_ckpt.sh $(grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' /content/files_tunnel.log | head -1) $CFG $EXP"


In [ ]:
%%bash
set -o pipefail
LOG="$ROOT/logs/${CFG}_${EXP}.log"
echo "== $(date) start, STEPS=$STEPS BATCH=$BATCH" >> "$LOG"
$RUN train $TASK --repo $REPO --root $ROOT --ckpt-root $CKPT_LOCAL --exp $EXP --steps $STEPS --batch-size $BATCH \
    --log-interval 100 --save-interval 1000 --keep-period $KEEP_PERIOD --resume 2>&1 \
  | $TIMER --steps $STEPS --quiet | tee -a "$LOG"
status=$?
pkill -f sync_ckpt.sh || true
/content/sync_ckpt.sh once
echo "training exit $status; on Drive:"; du -sh "$ROOT/checkpoints/$CFG/$EXP"/*

## Open-loop check

On the held-out `_val` demos: from every 15th frame, predict a chunk and compare its first 8 steps with the expert's
commands. `arm_mae_rad` should be well below `arm_hold_mae_rad` (the error of not moving), and the gripper error small.
This only checks imitation; success is measured in the sim. Set `STEP` to check an earlier checkpoint.

In [ ]:
STEP = None   # None = the latest
step_arg = f'--step {STEP}' if STEP is not None else ''
CK = CKPT_LOCAL if os.listdir(f'{CKPT_LOCAL}/{CFG}/{EXP}') else f'{ROOT}/checkpoints'   # Drive on a fresh runtime
!$RUN check $TASK --repo $REPO --root $ROOT --ckpt-root $CK --exp $EXP {step_arg} 2>&1 | grep -v -i warn | tail -20

## Serve for the sim

Starts openpi's websocket policy server on port 8000, then a Cloudflare quick tunnel to it (no account needed). Anyone
with the printed URL can query the policy while it runs. On the Mac, the shelf rollouts (`helpsim/pi05_shelf.py`) connect
to it:

```bash
source env.sh
$PY scripts/pi05_rollouts.py episodes --policy wss://<printed>.trycloudflare.com --out data/shelf_pnp/pi05_eval/run1     --objects tomato egg orange --episodes 10 --samples 5
```

On a fresh runtime (say, days later), run Settings and Install, then these cells: they load the checkpoint from Drive.
Stop with `pkill -f openpi_run.py; pkill -f localhost:8000`.

In [ ]:
%%bash
pkill -f "openpi_run.py serve" || true
CK=$CKPT_LOCAL; [ -n "$(ls -A $CKPT_LOCAL/$CFG/$EXP 2>/dev/null)" ] || CK=$ROOT/checkpoints   # Drive on a fresh runtime
nohup bash -c "$RUN serve $TASK --repo $REPO --root $ROOT --ckpt-root $CK --exp $EXP" > /content/serve.log 2>&1 &
for i in $(seq 1 120); do grep -q "serving" /content/serve.log && break; sleep 5; done
tail -2 /content/serve.log

In [ ]:
%%bash
CF=/content/cloudflared
[ -x $CF ] || { wget -q -O $CF https://github.com/cloudflare/cloudflared/releases/download/2025.8.1/cloudflared-linux-amd64 && chmod +x $CF; }
pkill -f 'localhost:8000' || true   # the policy's tunnel only
nohup $CF tunnel --no-autoupdate --url http://localhost:8000 > /content/tunnel.log 2>&1 &
for i in $(seq 1 30); do grep -qo 'https://[a-z0-9-]*\.trycloudflare\.com' /content/tunnel.log && break; sleep 2; done
URL=$(grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' /content/tunnel.log | head -1)
echo "policy server: ${URL/https/wss}"

## Next task

For the drawer: set `TASK = 'drawer'` and its `BUDGET_UNITS` in Settings, then run from Settings down (the install is
kept while the runtime is up).